# Bangladesh Urban Center Mapping — FINAL CLEAN VERSION

This notebook is a cleaned, duplicate-free, all-online workflow for national Bangladesh urban-center mapping.

## Data sources
- Sentinel-2: Element84 Earth Search STAC
- VIIRS night lights: Google Earth Engine
- GHSL population 2025: Google Earth Engine
- Copernicus DEM GLO-30: Google Earth Engine
- Roads: Overture Maps cloud GeoParquet
- POI / services: Overture Maps cloud GeoParquet

## Processing logic
Bangladesh AOI → Sentinel-2 composite → spectral built-up score → VIIRS → GHSL population → roads → POI/services → topography → common 100 m grid → weighted Urban Score → threshold → connected components → minimum patch / mean score / population-density filters → final urban-center polygons.

### Important design changes
- No Xee is used.
- Earth Engine factors are processed server-side and automatically cached by code as GeoTIFFs. This is not a manual data-download workflow.
- Overture roads are bbox-filtered in DuckDB and rasterized directly; millions of road segments are **not** exact-clipped with `gpd.clip()`.
- All final factors are local xarray rasters on the exact same EPSG:6933 / 100 m template before weighted overlay.


In [4]:
# 0. RECOMMENDED CLEAN ENVIRONMENT
#
# Run once in Anaconda Prompt:
#
# conda create -n urban_center -c conda-forge --override-channels ^
#   python=3.11 geopandas rasterio gdal proj pyproj rioxarray xarray dask ^
#   pystac-client stackstac leafmap shapely scipy matplotlib pandas numpy ^
#   pyogrio duckdb earthengine-api geemap geedim ipykernel -y
#
# conda activate urban_center
# python -m ipykernel install --user --name urban_center --display-name "Python (urban_center)"
#
# Then select "Python (urban_center)" as the VS Code notebook kernel.


In [5]:
# 1. CONFIGURATION

from pathlib import Path
import os

PROJECT_DIR = Path(r"E:\Geospatial\Urban Center\Urban-Center")
AOI_PATH = PROJECT_DIR / "bgd_admin_boundaries.shp" / "bgd_admin0.shp"

OUTPUT_DIR = PROJECT_DIR / "outputs"
CACHE_DIR = PROJECT_DIR / "cache_online"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Earth Engine project
EE_PROJECT = "solid-garden-458417-t4"

# Sentinel-2
S2_DATE_RANGE = "2025-01-01/2025-03-31"
MAX_CLOUD = 10
N_BEST_PER_TILE = 5

# Common national grid
TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

# Current Overture release used by this notebook.
# Change only this one value if Overture publishes a newer release.
OVERTURE_RELEASE = "2026-08-19.0"

# Multi-factor weights
WEIGHTS = {
    "builtup": 0.30,
    "nightlight": 0.20,
    "population": 0.20,
    "road": 0.10,
    "poi": 0.10,
    "topography": 0.10,
}

URBAN_SCORE_THRESHOLD = 0.55
MIN_PATCH_AREA_KM2 = 1.0
MIN_MEAN_POP_DENSITY = 500.0
MIN_MEAN_URBAN_SCORE = 0.60

# 1-km density neighborhood
DENSITY_WINDOW_M = 1000

# Remote COG retry settings
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["GDAL_HTTP_RETRY_CODES"] = "ALL"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

print("AOI:", AOI_PATH)
print("Outputs:", OUTPUT_DIR)
print("Online cache:", CACHE_DIR)
print("Overture release:", OVERTURE_RELEASE)


AOI: E:\Geospatial\Urban Center\Urban-Center\bgd_admin_boundaries.shp\bgd_admin0.shp
Outputs: E:\Geospatial\Urban Center\Urban-Center\outputs
Online cache: E:\Geospatial\Urban Center\Urban-Center\cache_online
Overture release: 2026-08-19.0


In [6]:
# 2. IMPORTS + ENVIRONMENT CHECK

from collections import defaultdict, Counter
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
from rasterio.errors import RasterioIOError
from rasterio.features import rasterize, shapes
from rasterio.enums import Resampling, MergeAlg

import pyproj
import xarray as xr
import rioxarray
import stackstac
from pystac_client import Client

from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely import from_wkb

from scipy import ndimage
import duckdb
import leafmap
import ee
import geemap

warnings.filterwarnings("ignore", category=FutureWarning)

print("Rasterio:", rasterio.__version__)
print("GDAL:", rasterio.__gdal_version__)
print("PROJ:", pyproj.proj_version_str)
print("StackSTAC:", stackstac.__version__)

try:
    print("CRS test:", rasterio.crs.CRS.from_epsg(TARGET_EPSG))
except Exception as exc:
    raise RuntimeError(
        "GDAL/PROJ environment mismatch. Use the clean urban_center environment "
        "from Cell 0 before continuing.\n"
        f"Original error: {exc}"
    )


Rasterio: 1.4.4
GDAL: 3.10.3
PROJ: 9.5.1
StackSTAC: 0.5.1


RuntimeError: GDAL/PROJ environment mismatch. Use the clean urban_center environment from Cell 0 before continuing.
Original error: The EPSG code is unknown. PROJ: proj_create_from_database: C:\Users\HP\.conda\envs\geo\Library\share\proj\proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 5 is expected. It comes from another PROJ installation.

In [7]:
# 3. LOAD BANGLADESH AOI

if not AOI_PATH.exists():
    raise FileNotFoundError(f"AOI not found: {AOI_PATH}")

bd = gpd.read_file(AOI_PATH)

if bd.empty:
    raise ValueError("Bangladesh AOI is empty.")
if bd.crs is None:
    raise ValueError("Bangladesh AOI has no CRS.")

bd = bd.to_crs(4326)
aoi = bd[["geometry"]].dissolve().reset_index(drop=True)

if not aoi.geometry.iloc[0].is_valid:
    aoi["geometry"] = aoi.geometry.buffer(0)

bd_geom = aoi.geometry.iloc[0]
west, south, east, north = map(float, aoi.total_bounds)

print("AOI CRS:", aoi.crs)
print("Bounds:", (west, south, east, north))
print("AOI valid:", bd_geom.is_valid)


AOI CRS: EPSG:4326
Bounds: (88.00816912400006, 20.590608254000188, 92.68005782300008, 26.634548266000024)
AOI valid: True


In [8]:
# 4. INTERACTIVE AOI MAP

m = leafmap.Map(center=[23.6850, 90.3563], zoom=7, height="650px")
m.add_gdf(
    aoi,
    layer_name="Bangladesh AOI",
    style={"color": "red", "weight": 3, "fillOpacity": 0.04},
)
m


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## A. Sentinel-2 — Earth Search STAC
The API search uses the Bangladesh bounding box to avoid oversized geometry requests, then filters footprints by the exact Bangladesh polygon locally.


In [9]:
# 5. SEARCH SENTINEL-2

catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=[west, south, east, north],
    datetime=S2_DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
)

items_raw = list(search.items())
print("Raw bbox results:", len(items_raw))

items_bd = []

for item in items_raw:
    if item.geometry is None:
        continue
    try:
        if shape(item.geometry).intersects(bd_geom):
            items_bd.append(item)
    except Exception:
        pass

if not items_bd:
    raise RuntimeError("No Sentinel-2 scenes intersect Bangladesh.")

print("Scenes intersecting Bangladesh:", len(items_bd))


Raw bbox results: 984
Scenes intersecting Bangladesh: 634


In [10]:
# 6. SELECT LOW-CLOUD SCENES + COVERAGE COMPLETION

def get_tile_id(item):
    parts = item.id.split("_")
    if len(parts) > 1 and parts[1].startswith("T"):
        return parts[1]
    grid_code = item.properties.get("grid:code")
    if grid_code:
        return str(grid_code)
    raise ValueError(item.id)

items_by_tile = defaultdict(list)

for item in items_bd:
    try:
        items_by_tile[get_tile_id(item)].append(item)
    except ValueError:
        pass

selected_items = []

for tile_id, tile_items in items_by_tile.items():
    tile_items = sorted(
        tile_items,
        key=lambda x: (
            x.properties.get("eo:cloud_cover", 100),
            x.datetime.isoformat() if x.datetime else "",
        ),
    )
    selected_items.extend(tile_items[:N_BEST_PER_TILE])

def footprint_union(stac_items):
    geoms = [shape(i.geometry) for i in stac_items if i.geometry is not None]
    if not geoms:
        raise RuntimeError("No valid footprints.")
    return unary_union(geoms)

selected_union = footprint_union(selected_items)
missing_geom = bd_geom.difference(selected_union)
selected_ids = {i.id for i in selected_items}

remaining_items = sorted(
    [i for i in items_bd if i.id not in selected_ids],
    key=lambda x: x.properties.get("eo:cloud_cover", 100),
)

extra_items = []

for item in remaining_items:
    if missing_geom.is_empty:
        break
    scene_geom = shape(item.geometry)
    gain = missing_geom.intersection(scene_geom)
    if not gain.is_empty and gain.area > 0:
        selected_items.append(item)
        extra_items.append(item)
        selected_union = selected_union.union(scene_geom)
        missing_geom = bd_geom.difference(selected_union)

coverage_check = gpd.GeoDataFrame(
    {"kind": ["AOI", "covered"]},
    geometry=[bd_geom, bd_geom.intersection(selected_union)],
    crs="EPSG:4326",
).to_crs(TARGET_EPSG)

coverage_pct = (
    coverage_check.geometry.iloc[1].area
    / coverage_check.geometry.iloc[0].area
    * 100.0
)

print("MGRS tiles:", len(items_by_tile))
print("Selected scenes:", len(selected_items))
print("Extra scenes:", len(extra_items))
print(f"Footprint coverage: {coverage_pct:.6f}%")


MGRS tiles: 36
Selected scenes: 187
Extra scenes: 7
Footprint coverage: 100.000000%


In [11]:
# 7. BUILD ONE RESILIENT SENTINEL STACK

REQUIRED_ASSETS = ["blue", "green", "red", "nir", "swir16", "scl"]

selected_items_clean = [
    item for item in selected_items
    if all(asset in item.assets for asset in REQUIRED_ASSETS)
]

if not selected_items_clean:
    raise RuntimeError("No selected scene has all required Sentinel assets.")

sentinel = stackstac.stack(
    selected_items_clean,
    assets=REQUIRED_ASSETS,
    bounds_latlon=[west, south, east, north],
    epsg=TARGET_EPSG,
    resolution=RESOLUTION_M,
    chunksize=CHUNK_SIZE,
    dtype=np.float32,
    fill_value=np.float32(np.nan),
    rescale=False,
    errors_as_nodata=(RasterioIOError(r".*"),),
)

print("Scenes stacked:", len(selected_items_clean))
print("Virtual stack:", sentinel.shape)


Scenes stacked: 187
Virtual stack: (187, 6, 7078, 4509)


In [12]:
# 8. CLOUD MASK + MEDIAN COMPOSITE + INDICES

scl = sentinel.sel(band="scl")
INVALID_SCL = [0, 1, 3, 8, 9, 10, 11]
valid = ~scl.isin(INVALID_SCL)

blue = sentinel.sel(band="blue").where(valid).median("time", skipna=True)
green = sentinel.sel(band="green").where(valid).median("time", skipna=True)
red = sentinel.sel(band="red").where(valid).median("time", skipna=True)
nir = sentinel.sel(band="nir").where(valid).median("time", skipna=True)
swir = sentinel.sel(band="swir16").where(valid).median("time", skipna=True)

EPS = np.float32(1e-6)

def safe_nd(a, b):
    den = a + b
    return ((a - b) / den.where(np.abs(den) > EPS)).clip(-1, 1).astype("float32")

ndbi = safe_nd(swir, nir).rename("NDBI")
ndvi = safe_nd(nir, red).rename("NDVI")
mndwi = safe_nd(green, swir).rename("MNDWI")

bsi_num = (swir + red) - (nir + blue)
bsi_den = (swir + red) + (nir + blue)
bsi = (bsi_num / bsi_den.where(np.abs(bsi_den) > EPS)).clip(-1, 1).astype("float32").rename("BSI")

def index01(da):
    return ((da + 1.0) / 2.0).clip(0, 1).astype("float32")

# Spectral built-up score.
# Higher NDBI, lower vegetation, lower water, and lower bare-soil response
# increase the built-up evidence.
builtup_probability = (
    0.50 * index01(ndbi)
    + 0.20 * (1.0 - index01(ndvi))
    + 0.20 * (1.0 - index01(mndwi))
    + 0.10 * (1.0 - index01(bsi))
).clip(0, 1).astype("float32").rename("Builtup_Probability")

print("Built-up factor prepared.")


Built-up factor prepared.


In [13]:
# 9. COMMON GRID HELPERS

template = (
    ndbi.astype("float32")
    .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
)

aoi_target = aoi.to_crs(TARGET_EPSG)

def add_rio_metadata(da):
    return (
        da.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

def clip_bd(da):
    da = add_rio_metadata(da)
    return da.rio.clip(
        aoi_target.geometry,
        aoi_target.crs,
        drop=True,
        all_touched=False,
    )

def align_to_template(da, method="bilinear"):
    resampling = {
        "nearest": Resampling.nearest,
        "bilinear": Resampling.bilinear,
    }[method]
    return da.rio.reproject_match(template, resampling=resampling).astype("float32")

def open_align_cache(path, method="bilinear"):
    da = (
        rioxarray.open_rasterio(path, masked=True, chunks="auto")
        .squeeze(drop=True)
        .astype("float32")
    )
    return align_to_template(da, method)

def robust_norm_local(da, p_low=2, p_high=98):
    arr = np.asarray(da.values)
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        raise ValueError(f"{da.name}: no finite values.")
    lo, hi = np.nanpercentile(vals, [p_low, p_high])
    if hi <= lo:
        return xr.zeros_like(da, dtype="float32")
    return ((da - lo) / (hi - lo)).clip(0, 1).astype("float32")

def np_to_template(arr, name):
    da = xr.DataArray(
        np.asarray(arr, dtype="float32"),
        dims=("y", "x"),
        coords={"y": template.y.values, "x": template.x.values},
        name=name,
    )
    return add_rio_metadata(da)

print("Template shape:", (template.sizes["y"], template.sizes["x"]))
print("Template CRS:", template.rio.crs)


Template shape: (7078, 4509)
Template CRS: PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","6933"]]


## B. Earth Engine factors — VIIRS, GHSL Population, Topography
These factors stay in Earth Engine for server-side preprocessing. The notebook then automatically caches each result as a GeoTIFF in EPSG:4326 and reprojects it locally to the Sentinel 100 m template. This avoids all Xee API/version issues.


In [14]:
# 10. EARTH ENGINE INITIALIZATION

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized.")
except Exception:
    print("Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized after authentication.")

ee_aoi = ee.Geometry(bd_geom.__geo_interface__)

def ee_percentile_norm(image, band, region, scale):
    image = ee.Image(image).select(band)
    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )
    lo = ee.Number(stats.get(f"{band}_p2"))
    hi = ee.Number(stats.get(f"{band}_p98"))
    return image.subtract(lo).divide(hi.subtract(lo).max(1e-6)).clamp(0, 1)

def cache_ee_image(image, filename, scale, resampling="bilinear"):
    """
    Automatically download/cache an EE result; no manual source-data download.
    Uses WGS84 for the transfer, then reprojects to the exact local template.
    """
    path = CACHE_DIR / filename

    if not path.exists():
        print("Caching Earth Engine image:", filename)
        geemap.download_ee_image(
            image=image,
            filename=str(path),
            region=ee_aoi,
            crs="EPSG:4326",
            scale=scale,
            resampling="bilinear" if resampling == "bilinear" else "near",
            overwrite=True,
            num_threads=4,
        )

    if not path.exists():
        raise RuntimeError(f"Earth Engine cache was not created: {path}")

    return open_align_cache(path, resampling)

print("EE helpers ready.")


Earth Engine initialized.
EE helpers ready.


In [ ]:
# 11. VIIRS NIGHT-LIGHT SCORE — ONLINE

viirs_col = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterDate("2025-01-01", "2026-01-01")
    .filterBounds(ee_aoi)
)

print("VIIRS monthly images:", viirs_col.size().getInfo())

def clean_viirs(img):
    rad = img.select("avg_rad")
    coverage = img.select("cf_cvg")
    return (
        rad.updateMask(coverage.gt(0))
        .max(0)
        .rename("avg_rad")
        .copyProperties(img, ["system:time_start"])
    )

viirs_clean = viirs_col.map(clean_viirs)

# Annual median gives a robust persistent-light intensity surface.
viirs_median = viirs_clean.median().rename("viirs_median").clip(ee_aoi)

nightlight_ee = (
    ee_percentile_norm(viirs_median.add(1).log().rename("light_log"), "light_log", ee_aoi, 500)
    .rename("NightLight_Score")
    .clip(ee_aoi)
)

nightlight_score = cache_ee_image(
    nightlight_ee,
    "VIIRS_NightLight_Score_2025.tif",
    scale=500,
    resampling="bilinear",
).rename("NightLight_Score")

print("nightlight  : READY")


In [ ]:
# 12. GHSL POPULATION 2025 — ONLINE

pop_count_ee = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2025")
    .select("population_count")
    .max(0)
    .clip(ee_aoi)
    .rename("population_count")
)

# GHSL population_count is count per 100 m cell.
# 100m x 100m = 0.01 km2 -> density ≈ count * 100.
pop_density_ee = pop_count_ee.multiply(100.0).rename("population_density")

pop_log_ee = pop_density_ee.add(1).log().rename("pop_log")

population_score_ee = (
    ee_percentile_norm(pop_log_ee, "pop_log", ee_aoi, 100)
    .rename("Population_Score")
    .clip(ee_aoi)
)

population_density = cache_ee_image(
    pop_density_ee,
    "GHSL_Population_Density_2025.tif",
    scale=100,
    resampling="bilinear",
).rename("Population_Density")

population_score = cache_ee_image(
    population_score_ee,
    "GHSL_Population_Score_2025.tif",
    scale=100,
    resampling="bilinear",
).rename("Population_Score")

print("population  : READY")


In [ ]:
# 13. COPERNICUS DEM + SLOPE + TOPOGRAPHY — ONLINE

dem_col = ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1")
native_projection = dem_col.first().select("DEM").projection()

dem_ee = (
    dem_col.select("DEM")
    .mosaic()
    .setDefaultProjection(native_projection)
    .rename("elevation")
    .clip(ee_aoi)
)

slope_ee = ee.Terrain.slope(dem_ee).rename("slope").clip(ee_aoi)

elev01_ee = ee_percentile_norm(dem_ee, "elevation", ee_aoi, 100).rename("elevation01")
slope01_ee = ee_percentile_norm(slope_ee, "slope", ee_aoi, 100).rename("slope01")

# Lower elevation + lower slope = higher generic urban-suitability score.
topography_ee = (
    ee.Image(1).subtract(elev01_ee).multiply(0.50)
    .add(ee.Image(1).subtract(slope01_ee).multiply(0.50))
    .clamp(0, 1)
    .rename("Topography_Score")
    .clip(ee_aoi)
)

topography_score = cache_ee_image(
    topography_ee,
    "Topography_Score_2025.tif",
    scale=100,
    resampling="bilinear",
).rename("Topography_Score")

print("topography  : READY")


## C. Overture Roads — direct cloud query
The query is bbox-filtered in DuckDB. We intentionally do **not** run `gpd.clip()` on ~1–2 million road lines. Rasterization occurs directly on the Bangladesh analysis grid, and the final raster is clipped by the exact AOI.


In [ ]:
# 14. OVERTURE / DUCKDB CONNECTION

con = duckdb.connect(database=":memory:")
con.execute("INSTALL spatial")
con.execute("LOAD spatial")
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET s3_region='us-west-2'")
con.execute("SET enable_object_cache=true")
con.execute("SET threads=4")

ROAD_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
)

POI_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=places/type=place/*"
)

print("DuckDB + Overture ready.")


In [ ]:
# 15. QUERY BANGLADESH ROADS

road_sql = f"""
SELECT
    id,
    class,
    ST_AsWKB(geometry) AS geom_wkb
FROM read_parquet('{ROAD_URL}', hive_partitioning=1)
WHERE
    subtype = 'road'
    AND class IN (
        'motorway', 'trunk', 'primary', 'secondary',
        'tertiary', 'residential', 'living_street',
        'unclassified', 'service'
    )
    AND bbox.xmin <= {east}
    AND bbox.xmax >= {west}
    AND bbox.ymin <= {north}
    AND bbox.ymax >= {south}
"""

print("Querying Overture roads online...")
roads_df = con.execute(road_sql).fetch_df()
print("Raw road records:", len(roads_df))

if roads_df.empty:
    raise RuntimeError("Overture road query returned no records.")

# DuckDB may return WKB as bytearray.
wkb_values = roads_df["geom_wkb"].apply(
    lambda x: bytes(x) if isinstance(x, bytearray) else x
)

road_geometry = from_wkb(wkb_values.to_numpy())

roads = gpd.GeoDataFrame(
    roads_df.drop(columns=["geom_wkb"]),
    geometry=road_geometry,
    crs="EPSG:4326",
)

roads = roads[
    roads.geometry.notna() & ~roads.geometry.is_empty
].copy()

# Fast exact-intersection filter: no expensive geometry cutting.
roads = roads[roads.geometry.intersects(bd_geom)].copy()

print("Road segments intersecting Bangladesh:", len(roads))


In [ ]:
# 16. ROAD DENSITY + INTERSECTION DENSITY -> ROAD SCORE

roads_m = roads.to_crs(TARGET_EPSG)

transform = template.rio.transform()
out_shape = (template.sizes["y"], template.sizes["x"])

road_presence = rasterize(
    [(geom, 1) for geom in roads_m.geometry if geom is not None and not geom.is_empty],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True,
)

window_cells = max(1, int(round(DENSITY_WINDOW_M / RESOLUTION_M)))

road_density_np = ndimage.uniform_filter(
    road_presence.astype("float32"),
    size=window_cells,
    mode="constant",
    cval=0,
)

# Junction proxy from repeated segment endpoints.
endpoint_keys = []

for geom in roads_m.geometry:
    if geom is None or geom.is_empty:
        continue

    geoms = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]

    for line in geoms:
        try:
            coords = list(line.coords)
            if len(coords) >= 2:
                endpoint_keys.append((round(coords[0][0]), round(coords[0][1])))
                endpoint_keys.append((round(coords[-1][0]), round(coords[-1][1])))
        except Exception:
            pass

endpoint_counts = Counter(endpoint_keys)

intersection_points = [
    Point(x, y)
    for (x, y), count in endpoint_counts.items()
    if count >= 3
]

intersection_raster = rasterize(
    [(p, 1) for p in intersection_points],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="float32",
    merge_alg=MergeAlg.add,
)

intersection_density_np = ndimage.uniform_filter(
    intersection_raster,
    size=window_cells,
    mode="constant",
    cval=0,
)

road_density = np_to_template(road_density_np, "Road_Density")
intersection_density = np_to_template(intersection_density_np, "Intersection_Density")

road_score = (
    0.70 * robust_norm_local(road_density)
    + 0.30 * robust_norm_local(intersection_density)
).clip(0, 1).astype("float32").rename("Road_Score")

print("Road intersections:", len(intersection_points))
print("road        : READY")


## D. Overture POI / Services
POIs are queried online and classified into four workflow groups: market/shopping, education, health, and commercial/services.


In [ ]:
# 17. QUERY + CLASSIFY BANGLADESH POIs

poi_sql = f"""
SELECT
    id,
    basic_category,
    CAST(taxonomy AS VARCHAR) AS taxonomy_text,
    ST_AsWKB(geometry) AS geom_wkb
FROM read_parquet('{POI_URL}', hive_partitioning=1)
WHERE
    bbox.xmin BETWEEN {west} AND {east}
    AND bbox.ymin BETWEEN {south} AND {north}
    AND (
        regexp_matches(
            lower(coalesce(basic_category, '')),
            'market|shop|mall|grocery|supermarket|school|college|university|hospital|clinic|medical|health|bank|office|restaurant|hotel|pharmacy|service|business'
        )
        OR regexp_matches(
            lower(coalesce(CAST(taxonomy AS VARCHAR), '')),
            'market|shopping|school|education|college|university|hospital|clinic|health|medical|commercial|business|bank|office|restaurant|hotel|pharmacy|service'
        )
    )
"""

print("Querying Overture POIs online...")
poi_df = con.execute(poi_sql).fetch_df()
print("Raw POI records:", len(poi_df))

if poi_df.empty:
    raise RuntimeError("Overture POI query returned no records.")

poi_wkb = poi_df["geom_wkb"].apply(
    lambda x: bytes(x) if isinstance(x, bytearray) else x
)

poi_geoms = from_wkb(poi_wkb.to_numpy())

pois = gpd.GeoDataFrame(
    poi_df.drop(columns=["geom_wkb"]),
    geometry=poi_geoms,
    crs="EPSG:4326",
)

pois = pois[
    pois.geometry.notna()
    & ~pois.geometry.is_empty
    & pois.geometry.intersects(bd_geom)
].copy()

def classify_poi(row):
    text = (
        str(row.get("basic_category", ""))
        + " "
        + str(row.get("taxonomy_text", ""))
    ).lower()

    if any(k in text for k in ["school", "college", "university", "education", "academy"]):
        return "education"

    if any(k in text for k in ["hospital", "clinic", "medical", "health", "pharmacy", "doctor"]):
        return "health"

    if any(k in text for k in ["market", "supermarket", "grocery", "mall", "shopping", "shop"]):
        return "market"

    return "commercial_service"

pois["poi_group"] = pois.apply(classify_poi, axis=1)

print("POIs intersecting Bangladesh:", len(pois))
display(pois["poi_group"].value_counts().to_frame("count"))


In [ ]:
# 18. POI DENSITY -> POI/SERVICE SCORE

pois_m = pois.to_crs(TARGET_EPSG)

group_scores = []

for group in ["market", "education", "health", "commercial_service"]:
    subset = pois_m[pois_m["poi_group"] == group]

    point_raster = rasterize(
        [(geom, 1) for geom in subset.geometry if geom is not None and not geom.is_empty],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="float32",
        merge_alg=MergeAlg.add,
    )

    density_np = ndimage.uniform_filter(
        point_raster,
        size=window_cells,
        mode="constant",
        cval=0,
    )

    group_da = np_to_template(density_np, f"{group}_density")
    group_scores.append(robust_norm_local(group_da))

poi_score = (
    sum(group_scores) / len(group_scores)
).clip(0, 1).astype("float32").rename("POI_Service_Score")

print("poi         : READY")


## E. Final factor alignment and Urban Score
At this point **all six factors are local xarray rasters**. This avoids the original notebook's Earth Engine image vs xarray variable mismatch.


In [ ]:
# 19. ALIGN + CHECK ALL SIX FACTORS

factor_raw = {
    "builtup": builtup_probability,
    "nightlight": nightlight_score,
    "population": population_score,
    "road": road_score,
    "poi": poi_score,
    "topography": topography_score,
}

aligned_factors = {}

for name, da in factor_raw.items():
    da = add_rio_metadata(da)

    # Already on template: builtup, road, poi.
    # EE caches were aligned when opened; reproject_match again is harmless
    # and guarantees exact x/y identity.
    method = "nearest" if name in [] else "bilinear"
    aligned = align_to_template(da, method)
    aligned_factors[name] = aligned.clip(0, 1).astype("float32")

print("FACTOR STATUS")
print("-" * 35)

for name, da in aligned_factors.items():
    print(
        f"{name:12s}: READY",
        da.shape,
        da.rio.crs,
    )

print("\nALL SIX FACTORS ARE READY.")


In [ ]:
# 20. MULTI-FACTOR URBAN SCORE + EXACT BANGLADESH MASK

urban_score = sum(
    WEIGHTS[name] * aligned_factors[name]
    for name in WEIGHTS
).clip(0, 1).astype("float32").rename("Urban_Score")

urban_score_bd = clip_bd(urban_score)

urban_mask = (
    urban_score_bd >= URBAN_SCORE_THRESHOLD
).astype("uint8").rename("Urban_Center_Mask")

print("Urban Score READY")
print("Threshold:", URBAN_SCORE_THRESHOLD)


In [ ]:
# 21. CONNECTED COMPONENTS + MINIMUM PATCH FILTER

mask_np = urban_mask.compute().values.astype(bool)

labels, n_components = ndimage.label(
    mask_np,
    structure=np.ones((3, 3), dtype="uint8"),
)

pixel_area_km2 = (RESOLUTION_M * RESOLUTION_M) / 1_000_000.0
min_pixels = max(1, int(np.ceil(MIN_PATCH_AREA_KM2 / pixel_area_km2)))

counts = np.bincount(labels.ravel())
keep_labels = np.where(counts >= min_pixels)[0]
keep_labels = keep_labels[keep_labels != 0]

filtered_np = np.isin(labels, keep_labels)

filtered_mask = xr.DataArray(
    filtered_np.astype("uint8"),
    dims=("y", "x"),
    coords={"y": urban_mask.y.values, "x": urban_mask.x.values},
    name="Urban_Center_Filtered",
)
filtered_mask = add_rio_metadata(filtered_mask)

print("Initial components:", n_components)
print("Retained components:", len(keep_labels))
print("Minimum patch:", MIN_PATCH_AREA_KM2, "km2")


In [ ]:
# 22. POLYGONIZE + ZONAL MEANS

records = []

for geom, value in shapes(
    filtered_mask.values,
    mask=filtered_mask.values.astype(bool),
    transform=filtered_mask.rio.transform(),
):
    if int(value) == 1:
        records.append({"geometry": shape(geom)})

urban_polygons = gpd.GeoDataFrame(
    records,
    crs=f"EPSG:{TARGET_EPSG}",
)

if urban_polygons.empty:
    final_urban_centers = urban_polygons.copy()
    print("No urban-center candidate polygons.")
else:
    urban_polygons["area_km2"] = urban_polygons.geometry.area / 1_000_000.0
    urban_polygons = urban_polygons.reset_index(drop=True)

    pop_density_bd = clip_bd(population_density)

    def zonal_mean(gdf, da, field_name):
        arr = da.compute().values
        tr = da.rio.transform()
        means = []

        for geom in gdf.geometry:
            zone = rasterize(
                [(geom, 1)],
                out_shape=arr.shape,
                transform=tr,
                fill=0,
                dtype="uint8",
                all_touched=False,
            ).astype(bool)

            vals = arr[zone]
            vals = vals[np.isfinite(vals)]
            means.append(float(vals.mean()) if vals.size else np.nan)

        gdf[field_name] = means
        return gdf

    urban_polygons = zonal_mean(urban_polygons, urban_score_bd, "mean_score")
    urban_polygons = zonal_mean(urban_polygons, pop_density_bd, "mean_popden")

    final_urban_centers = urban_polygons[
        (urban_polygons["area_km2"] >= MIN_PATCH_AREA_KM2)
        & (urban_polygons["mean_score"] >= MIN_MEAN_URBAN_SCORE)
        & (urban_polygons["mean_popden"] >= MIN_MEAN_POP_DENSITY)
    ].copy()

    final_urban_centers = final_urban_centers.reset_index(drop=True)
    final_urban_centers["urban_id"] = np.arange(1, len(final_urban_centers) + 1)

    print("Candidate polygons:", len(urban_polygons))
    print("Final urban centers:", len(final_urban_centers))


In [ ]:
# 23. PREVIEW FINAL OUTPUT

preview = (
    urban_score_bd
    .coarsen(x=4, y=4, boundary="trim")
    .mean(skipna=True)
    .compute()
)

plt.figure(figsize=(8, 9))
preview.plot(vmin=0, vmax=1)
plt.title("Bangladesh Multi-Factor Urban Score")
plt.axis("equal")
plt.show()

final_map = leafmap.Map(center=[23.6850, 90.3563], zoom=7, height="700px")
final_map.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "black", "weight": 2, "fillOpacity": 0},
)

if not final_urban_centers.empty:
    final_map.add_gdf(
        final_urban_centers.to_crs(4326),
        layer_name="Final Urban Centers",
        style={"color": "red", "weight": 1, "fillColor": "red", "fillOpacity": 0.45},
    )

final_map


In [ ]:
# 24. EXPORT IMPORTANT OUTPUTS

export_rasters = {
    "Builtup_Probability_2025_100m.tif": clip_bd(aligned_factors["builtup"]),
    "NightLight_Score_2025_100m.tif": clip_bd(aligned_factors["nightlight"]),
    "Population_Score_2025_100m.tif": clip_bd(aligned_factors["population"]),
    "Road_Score_2025_100m.tif": clip_bd(aligned_factors["road"]),
    "POI_Service_Score_2025_100m.tif": clip_bd(aligned_factors["poi"]),
    "Topography_Score_2025_100m.tif": clip_bd(aligned_factors["topography"]),
    "Urban_Score_2025_100m.tif": urban_score_bd,
    "Urban_Center_Mask_2025_100m.tif": filtered_mask,
}

for filename, da in export_rasters.items():
    path = OUTPUT_DIR / filename
    print("Writing:", path)
    da.rio.to_raster(
        path,
        compress="DEFLATE",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

gpkg_path = OUTPUT_DIR / "Bangladesh_Urban_Centers_2025.gpkg"

if final_urban_centers.empty:
    print("No final polygons to write.")
else:
    final_urban_centers.to_file(
        gpkg_path,
        layer="urban_centers",
        driver="GPKG",
    )
    print("Final vector:", gpkg_path)

print("DONE")


## Run order
Run the notebook from top to bottom. Do not mix cells from the old notebook.

If a cloud query fails, rerun only that specific data-source cell after checking internet/authentication. The expensive Overture road query does not need to be repeated if `roads` is already present in the current kernel.
